In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load your data
directory = "C:\\Users\\20234503\\Desktop\\Research\\Weather and Strikes\\"
df = pd.read_csv(directory + "Data\\Weather Data\\Weather_Labour_Unrest_Event.csv")

# ---------- 0) Clean columns ----------
df.columns = (df.columns
                .str.strip()
                .str.lower()
                .str.replace(r"\s+", "_", regex=True))

# ---------- 1) Season variable ----------
if 'month' not in df.columns:
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['month'] = df['date'].dt.month
    else:
        raise KeyError("Need 'date' or 'month' to compute seasons.")

df['season'] = np.select(
    [
        df['month'].isin([12, 1, 2]),
        df['month'].isin([3, 4, 5]),
        df['month'].isin([6, 7, 8]),
        df['month'].isin([9, 10, 11]),
    ],
    [1, 2, 3, 4],
    default=np.nan
)

code_to_name = {1:'Winter', 2:'Spring', 3:'Summer', 4:'Fall'}
df['season_name'] = df['season'].map(code_to_name)
df['season_name'] = pd.Categorical(
    df['season_name'],
    categories=['Winter','Spring','Summer','Fall'],
    ordered=True
)

# ---------- 3) Scale to annualised per 100,000 firms ----------
scale = 100_000 * 365

to_scale = [
    'labour_unrest', 'strike', 'protest',
]

for v in to_scale:
    if v in df.columns:
        df[f'{v}_est_100_annual'] = (
            pd.to_numeric(df[v], errors='coerce') /
            pd.to_numeric(df['est'], errors='coerce')
        ) * scale

# ---------- 4) Row labels (pretty text only used for printing) ----------
nice = {
    'labour_unrest_est_100_annual': r'\hspace{0.05cm} Labour Conflict',
    'strike_est_100_annual':        r'\hspace{0.25cm} Strikes',
    'protest_est_100_annual':       r'\hspace{0.25cm} Protests',
}

# Order for Panel A, then Panel B
row_order_panel_a = [
    'labour_unrest_est_100_annual',
    'strike_est_100_annual','protest_est_100_annual',
]
vars_est = [c for c in row_order_panel_a if c in df.columns]

# ---------- 5) Summary means for Panel A ----------
def mean_only(df_sub, cols):
    cols = [c for c in cols if c in df_sub.columns]
    return pd.DataFrame({'mean': df_sub[cols].mean(numeric_only=True)})

summaries_a = {'All': mean_only(df, vars_est)}
for s in ['Winter','Spring','Summer','Fall']:
    summaries_a[s] = mean_only(df[df['season_name'] == s], vars_est)

means_panel_a = pd.concat(
    [summaries_a[k]['mean'].rename(k) for k in ['All','Winter','Spring','Summer','Fall']],
    axis=1
).loc[vars_est]

means_panel_a_fmt = means_panel_a.applymap(lambda x: f"{x:.2f}")

#---------- 7) Export LaTeX ----------
save_dir = r"C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2"
os.makedirs(save_dir, exist_ok=True)
out_path = os.path.join(save_dir, "summary_stat_labour_unrest.tex")

season_display_order = [
    ('All','All'),
    ('Winter','Winter'),
    ('Spring','Spring'),
    ('Summer','Summer'),
    ('Fall','Fall')
]

header_cols = "l" + "c" * len(season_display_order)
season_cols = [name for name, _ in season_display_order]  # ['All','Winter','Spring','Summer','Autumn']

# Labels for Panel A
label_by_key_a = {k: nice.get(k, k) for k in vars_est}

lines = []
lines += [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Summary of Labour Conflict Data}",
    r"\label{tab:summary_stat_LC}",
    r"\begin{tabular}{" + header_cols + r"}",
    r"\toprule",
    r" & " + " & ".join([hdr for _, hdr in season_display_order]) + r" \\",
    r"\midrule"
]

# Group headers
panel_span = str(1 + len(season_display_order))
group_headers = {
    'A':  rf"\multicolumn{{{panel_span}}}{{l}}{{\emph{{Annualised Incidence per 100{{,}}000 Firms}}}} \\",
}

# Panel A
lines.append(group_headers['A'])
for key in vars_est:
    label = label_by_key_a[key]
    row_vals = means_panel_a_fmt.loc[key]
    vals = " & ".join([row_vals.get(col, "") for col in season_cols])
    lines.append(label + " & " + vals + r" \\")

lines.append(r"\addlinespace")

lines += [r"\bottomrule", r"\end{tabular}", "\\begin{minipage}{0.7\linewidth}",
    "\\tiny",
    "\\setstretch{0.9}",
    "\\textit{Notes:} ",
    "This table presents summary statistics of county-day panel of labour conflict."
    "Panel A reports the annualised incidence of labour conflict events per 100,000 firms, and by action type (strikes and protests) for the full sample and across meteorological seasons.",
    "\\end{minipage}",
    "\\end{table}" ]

with open(out_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"LaTeX table saved to:\n{out_path}")


<>:134: SyntaxWarning: invalid escape sequence '\l'
<>:134: SyntaxWarning: invalid escape sequence '\l'
C:\Users\20234503\AppData\Local\Temp\ipykernel_20852\3012394615.py:134: SyntaxWarning: invalid escape sequence '\l'
  lines += [r"\bottomrule", r"\end{tabular}", "\\begin{minipage}{0.7\linewidth}",


LaTeX table saved to:
C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2\summary_stat_labour_unrest.tex


C:\Users\20234503\AppData\Local\Temp\ipykernel_20852\3012394615.py:85: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  means_panel_a_fmt = means_panel_a.applymap(lambda x: f"{x:.2f}")
